# Notebook 04 — Dataset Validation

## Objective

Before preprocessing or model training, it is essential to verify the integrity and consistency of the PANORAMA dataset.

This notebook performs quality assessment of all CT volumes and segmentation masks.

The following checks are performed:

- CT and segmentation availability
- Shape consistency
- Image spacing consistency
- Origin consistency
- Direction consistency
- Label integrity
- Dataset-wide validation statistics

The results are saved as **validation_report.csv** for later reference.

In [1]:
# ============================================================
# Project Setup
# ============================================================

import os
import sys
from pathlib import Path

# Move from notebooks/ -> project root
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

print("Project Root:")
print(PROJECT_ROOT)

print("\nCurrent Working Directory:")
print(Path.cwd())

Project Root:
d:\Pancreatic_Cancer_Thesis

Current Working Directory:
d:\Pancreatic_Cancer_Thesis


In [2]:
import src.visualization

dir(src.visualization)

['__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'describe_case',
 'find_tumor_slices',
 'np',
 'overlay_mask',
 'plt',
 'print_case_info',
 'random_case',
 'save_figure',
 'show_case',
 'show_middle_slice',
 'show_random_case',
 'show_slice',
 'window_image']

In [3]:
# ============================================================
# Imports
# ============================================================

from importlib import reload

import pandas as pd
import matplotlib.pyplot as plt

import src.io
import src.validation
import src.visualization

reload(src.io)
reload(src.validation)
reload(src.visualization)

# -------- IO --------
from src.io import (
    print_dataset_summary,
    list_ct_cases,
    load_case,
    random_case,
    load_metadata,
)

# -------- Validation --------
from src.validation import (
    validate_case,
    validate_dataset,
    generate_validation_report,
)

# -------- Visualization --------
from src.visualization import (
    show_case,
    print_case_info,
)

# -------- Config --------
from src.config import OUTPUTS_DIR

print("Modules loaded successfully.")

Modules loaded successfully.


In [4]:
print_dataset_summary()

PANORAMA Dataset
CT volumes          : 557
Manual labels       : 482
Automatic labels    : 1756
Metadata rows       : 2238


## Validate Entire Dataset

The following section validates every CT volume in the PANORAMA dataset.

For each case we verify

- Shape agreement
- Spacing agreement
- Origin agreement
- Direction agreement

The results are collected into a DataFrame for later analysis.

In [5]:
validation_df = validate_dataset()

validation_df.head()

25/557
50/557
75/557
100/557
125/557
150/557
175/557
200/557
225/557
250/557
275/557
300/557
325/557
350/557
375/557
400/557
425/557
450/557
475/557
500/557
525/557
550/557
557/557


,study_id,mask_type,valid,error,shape_match,spacing_match,origin_match,direction_match,ct_size,mask_size,ct_spacing,mask_spacing,ct_origin,mask_origin
0,100000_00001,Automatic,True,,True,True,True,True,"(512, 512, 90)","(512, 512, 90)","(0.6949999928474426, 0.6949999928474426, 2.399...","(0.6949999928474426, 0.6949999928474426, 2.399...","(-181.6833038330078, -177.777099609375, 1759.4...","(-181.6833038330078, -177.777099609375, 1759.4..."
1,100001_00001,Automatic,True,,True,True,True,True,"(512, 512, 105)","(512, 512, 105)","(0.78125, 0.78125, 4.0)","(0.78125, 0.78125, 4.0)","(-184.609375, -347.609375, -248.5)","(-184.609375, -347.609375, -248.5)"
2,100002_00001,Manual,True,,True,True,True,True,"(512, 512, 561)","(512, 512, 561)","(0.7089999914169312, 0.7089999914169312, 1.0)","(0.7089999914169312, 0.7089999914169312, 1.0)","(-208.62960815429688, -164.68431091308594, 191...","(-208.62960815429688, -164.68431091308594, 191..."
3,100003_00001,Automatic,True,,True,True,True,True,"(512, 512, 101)","(512, 512, 101)","(0.6980000138282776, 0.6980000138282776, 2.400...","(0.6980000138282776, 0.6980000138282776, 2.400...","(-191.05679321289062, -178.55679321289062, 170...","(-191.05679321289062, -178.55679321289062, 170..."
4,100004_00001,Automatic,True,,True,True,True,True,"(512, 512, 163)","(512, 512, 163)","(0.85546875, 0.85546875, 1.5)","(0.85546875, 0.85546875, 1.5)","(-245.572265625, -451.572265625, 851.099975585...","(-245.572265625, -451.572265625, 851.099975585..."


In [6]:
validation_df

,study_id,mask_type,valid,error,shape_match,spacing_match,origin_match,direction_match,ct_size,mask_size,ct_spacing,mask_spacing,ct_origin,mask_origin
0,100000_00001,Automatic,True,,True,True,True,True,"(512, 512, 90)","(512, 512, 90)","(0.6949999928474426, 0.6949999928474426, 2.399...","(0.6949999928474426, 0.6949999928474426, 2.399...","(-181.6833038330078, -177.777099609375, 1759.4...","(-181.6833038330078, -177.777099609375, 1759.4..."
1,100001_00001,Automatic,True,,True,True,True,True,"(512, 512, 105)","(512, 512, 105)","(0.78125, 0.78125, 4.0)","(0.78125, 0.78125, 4.0)","(-184.609375, -347.609375, -248.5)","(-184.609375, -347.609375, -248.5)"
2,100002_00001,Manual,True,,True,True,True,True,"(512, 512, 561)","(512, 512, 561)","(0.7089999914169312, 0.7089999914169312, 1.0)","(0.7089999914169312, 0.7089999914169312, 1.0)","(-208.62960815429688, -164.68431091308594, 191...","(-208.62960815429688, -164.68431091308594, 191..."
3,100003_00001,Automatic,True,,True,True,True,True,"(512, 512, 101)","(512, 512, 101)","(0.6980000138282776, 0.6980000138282776, 2.400...","(0.6980000138282776, 0.6980000138282776, 2.400...","(-191.05679321289062, -178.55679321289062, 170...","(-191.05679321289062, -178.55679321289062, 170..."
4,100004_00001,Automatic,True,,True,True,True,True,"(512, 512, 163)","(512, 512, 163)","(0.85546875, 0.85546875, 1.5)","(0.85546875, 0.85546875, 1.5)","(-245.572265625, -451.572265625, 851.099975585...","(-245.572265625, -451.572265625, 851.099975585..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
552,100542_00001,Manual,True,,True,True,True,True,"(512, 512, 85)","(512, 512, 85)","(0.7324219942092896, 0.7324219942092896, 2.5)","(0.7324219942092896, 0.7324219942092896, 2.5)","(374.26763916015625, 374.26763916015625, 0.0)","(374.26763916015625, 374.26763916015625, 0.0)"
553,100543_00001,Automatic,True,,True,True,True,True,"(512, 512, 206)","(512, 512, 206)","(0.8984375, 0.8984375, 1.0)","(0.8984375, 0.8984375, 1.0)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)"
554,100544_00001,Manual,True,,True,True,True,True,"(512, 512, 128)","(512, 512, 128)","(0.78125, 0.78125, 5.0)","(0.78125, 0.78125, 5.0)","(-200.0, -22.0, -690.5)","(-200.0, -22.0, -690.5)"
555,100545_00001,Manual,True,,True,True,True,True,"(512, 512, 615)","(512, 512, 615)","(0.7689999938011169, 0.7689999938011169, 1.0)","(0.7689999938011169, 0.7689999938011169, 1.0)","(-196.4904022216797, -196.4904022216797, 1938.5)","(-196.4904022216797, -196.4904022216797, 1938.5)"


In [7]:
validation_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 557 entries, 0 to 556
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   study_id         557 non-null    str   
 1   mask_type        557 non-null    str   
 2   valid            557 non-null    bool  
 3   error            557 non-null    str   
 4   shape_match      557 non-null    bool  
 5   spacing_match    557 non-null    bool  
 6   origin_match     557 non-null    bool  
 7   direction_match  557 non-null    bool  
 8   ct_size          557 non-null    object
 9   mask_size        557 non-null    object
 10  ct_spacing       557 non-null    object
 11  mask_spacing     557 non-null    object
 12  ct_origin        557 non-null    object
 13  mask_origin      557 non-null    object
dtypes: bool(5), object(6), str(3)
memory usage: 42.0+ KB


## Generate Validation Report

In [8]:
summary = generate_validation_report(
    validation_df,
    OUTPUTS_DIR / "validation_report.csv",
)

PANORAMA DATASET VALIDATION REPORT
total_cases              : 557
valid_cases              : 555
invalid_cases            : 2
shape_mismatches         : 0
spacing_mismatches       : 1
origin_mismatches        : 0
direction_mismatches     : 1

Saved report to:
D:\Pancreatic_Cancer_Thesis\outputs\validation_report.csv


In [9]:
summary

{'total_cases': 557,
 'valid_cases': 555,
 'invalid_cases': 2,
 'shape_mismatches': 0,
 'spacing_mismatches': 1,
 'origin_mismatches': 0,
 'direction_mismatches': 1}